In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
apikey = os.getenv("GOOGLE_API_KEY")
if apikey:
    print("yes")
else:
    print("no")



yes


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI as Gai
llm = Gai(
    model="gemini-2.5-flash-lite",
    temperature=0.5,
    max_output_tokens=300,
    top_k=3   
)

In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a math expert.

Reason carefully before answering.
Keep reasoning concise and structured.
"""),
    
    ("human", """
Example:

Question:
What is 10 × 2?

Answer:
Step 1: Multiply 10 by 2
Step 2: 10 × 2 = 20
Final Answer: 20

---

Now solve the problem below.

Question:
{question}

Answer:
Step 1:
Step 2:
Step 3:
Final Answer:
""")
])

In [4]:
chain = prompt | llm

response = chain.invoke({
    "question": "What is 25 × 4?"
})

print(response.content)

Step 1: Multiply 25 by 4.
Step 2: 25 × 4 = 100
Step 3: The result of the multiplication is 100.
Final Answer: 100


In [ ]:
# pip install wikipedia arxiv langchain-community

from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_community.tools.arxiv.tool import ArxivQueryRun
from langchain_community.utilities.arxiv import ArxivAPIWrapper

# Initialize the wrappers (No API keys required!)
wiki_wrapper = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1000)
arxiv_wrapper = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=1000)

# Create the tools
wikipedia_tool = WikipediaQueryRun(api_wrapper=wiki_wrapper)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

# These tools can now be directly passed into an agent
tools = [wikipedia_tool, arxiv_tool]


 agent


In [ ]:
query = input()
tool_output = arxiv_tool.run(query)
prompt = f"""
You are a research assistant. Answer the user's query based on the provided arXiv research papers.

User Query: 
"{query}"

Source Papers:
```text
{tool_output}"""

response = llm.invoke(prompt)
print(response.content)




Based on the provided arXiv paper, there is no information about whether ML models are going to die. The paper discusses the Mercedes-Benz GLE SUV and its history.


In [14]:
query = input("Enter your question: ")

# 1. Router Step
router_prompt = f"""
Decide if a Wikipedia search is absolutely necessary to answer this question accurately.
For general knowledge, history, or specific facts, choose YES. For logic, math, formatting, or creative tasks, choose NO.

Question: {query}

Reply with exactly one word: YES or NO.
"""

# Clean the decision string by removing punctuation and whitespace
decision = llm.invoke(router_prompt).content.strip().lower().replace(".", "")

# 2. Tool Execution & Dynamic Prompting
if "yes" in decision:
    tool_output1 = wikipedia_tool.run(query)
    
    prompt1 = f"""
    You are a helpful research assistant.
    Answer the user's question ONLY using the provided Wikipedia information. 
    If the information is missing or insufficient to answer, state exactly that.

    User Question: "{query}"
    Wikipedia Information: {tool_output1}
    """
else:
    # Fallback: If no tool is needed, let the LLM use its base knowledge directly
    prompt1 = f"""
    You are a helpful assistant. Answer the user's question concisely.

    User Question: "{query}"
    """

# 3. Final Answer Generation
response1 = llm.invoke(prompt1)

print("\n--- ROUTER DECISION ---")
print(f"Wikipedia Used: {decision.upper()}")
print("\n--- FINAL ANSWER ---")
print(response1) # Assuming you are using LangChain/similar where .content gets the text

KeyboardInterrupt: 

In [12]:
print(f"Total tokens used: {response.usage_metadata['total_tokens']}")

Total tokens used: 171


Conditional Tool Calling
